# LIGER best - cellranger_versions

Seeds: `1-10`

Metrics: BAMI, CASW, SCGRAPHM, BRAS, KBET, CPCR


In [1]:
import gc
import importlib.util
import os
import sys

os.environ["PYTHONNOUSERSITE"] = "1"
sys.path = [
    path
    for path in sys.path
    if ".local/lib/python3.10/site-packages" not in path
]

PROJECT_ROOT = '/ubc/cs/research/beaver/projects/Single-cell-integration-metrics'
os.environ["R_HOME"] = f"{PROJECT_ROOT}/CondaEnv/pyliger_env/lib/R"
sys.path.insert(0, f"{PROJECT_ROOT}/scib/")

import anndata as ad
import numpy as np
import pandas as pd

OPTIMIZER_PATH = (
    f"{PROJECT_ROOT}/scripts/optuna_multi_metrics/"
    "optuna_liger/optimize_liger_multi_metrics.py"
)
optimizer_spec = importlib.util.spec_from_file_location(
    "liger_opt",
    OPTIMIZER_PATH,
)
if optimizer_spec is None or optimizer_spec.loader is None:
    raise ImportError(f"Cannot load LIGER optimizer: {OPTIMIZER_PATH}")
liger_opt = importlib.util.module_from_spec(optimizer_spec)
sys.modules[optimizer_spec.name] = liger_opt
optimizer_spec.loader.exec_module(liger_opt)


/ubc/cs/research/beaver/projects/Single-cell-integration-metrics/CondaEnv/pyliger_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
DATASET_NAME = 'cellranger_versions'
DATA_PATH = f"{PROJECT_ROOT}/data/{DATASET_NAME}.h5ad"
BATCH_KEY = 'batch'
LABEL_KEY = 'cell_type'

SELECTED_BIO_METRICS = ('BAMI', 'CASW', 'SCGRAPHM')
SELECTED_BATCH_METRICS = ('BRAS', 'KBET', 'CPCR')
SELECTED_METRICS = SELECTED_BIO_METRICS + SELECTED_BATCH_METRICS

SEEDS = tuple(range(1, 11))
MODE = "core"
LAYER = "counts"
N_NEIGHBORS = 15
EMBED_KEY = "X_liger"

liger_opt.DATASET_NAME = DATASET_NAME
liger_opt.DATA_PATH = DATA_PATH
liger_opt.BATCH_KEY = BATCH_KEY
liger_opt.LABEL_KEY = LABEL_KEY
liger_opt.MODE = MODE
liger_opt.LAYER = LAYER
liger_opt.USE_BAMI = "BAMI" in SELECTED_METRICS
liger_opt.USE_CASW = "CASW" in SELECTED_METRICS
liger_opt.USE_SCGRAPHM = "SCGRAPHM" in SELECTED_METRICS
liger_opt.USE_TRAJ = "TRAJ" in SELECTED_METRICS
liger_opt.USE_ISOLISI = "ISOLISI" in SELECTED_METRICS
liger_opt.USE_BRAS = "BRAS" in SELECTED_METRICS
liger_opt.USE_KBET = "KBET" in SELECTED_METRICS
liger_opt.USE_CPCR = "CPCR" in SELECTED_METRICS

assert DATASET_NAME == 'cellranger_versions'
assert BATCH_KEY == 'batch'
assert LABEL_KEY == 'cell_type'
assert SELECTED_METRICS == ('BAMI', 'CASW', 'SCGRAPHM', 'BRAS', 'KBET', 'CPCR')
assert SEEDS == tuple(range(1, 11))
assert liger_opt.CORE_MAX_ITERS == 30
assert liger_opt.N_NEIGHBORS == N_NEIGHBORS
assert liger_opt.EMBED_KEY == EMBED_KEY

pd.Series(
    {
        "dataset": DATASET_NAME,
        "data_path": DATA_PATH,
        "batch_key": BATCH_KEY,
        "label_key": LABEL_KEY,
        "metrics": list(SELECTED_METRICS),
        "seeds": list(SEEDS),
        "mode": MODE,
        "layer": LAYER,
    },
    name="configuration",
)


dataset                                    cellranger_versions
data_path    /ubc/cs/research/beaver/projects/Single-cell-i...
batch_key                                                batch
label_key                                            cell_type
metrics               [BAMI, CASW, SCGRAPHM, BRAS, KBET, CPCR]
seeds                          [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
mode                                                      core
layer                                                   counts
Name: configuration, dtype: object

In [3]:
liger_opt.validate_metric_dependencies()
liger_opt.validate_configuration(list(SELECTED_METRICS))
liger_opt.validate_dataset_contents()


Dataset preflight: n_obs=19202 n_batches=2 n_labels=15
CPCR preflight: eligible_labels=12


In [4]:
OPTUNA_STUDY_NAME = 'liger_cellranger_versions_joint_expanded_metrics_scgraphm_v1'
OPTUNA_BEST_TRIAL = 99
OPTUNA_BEST_VALUE = 0.8281505136576194
OPTUNA_BEST_PARAMS = {'k': 20, 'value_lambda': 2.8640197425940483}

LIGER_BEST_ARGS = {
    "k": OPTUNA_BEST_PARAMS["k"],
    "value_lambda": OPTUNA_BEST_PARAMS["value_lambda"],
    "max_iters": 30,
    "mode": "core",
    "layer": "counts",
}


def run_liger_best(adata, seed):
    return liger_opt.run_liger(
        adata=adata,
        k=LIGER_BEST_ARGS["k"],
        lam=LIGER_BEST_ARGS["value_lambda"],
        max_iterations=LIGER_BEST_ARGS["max_iters"],
        mode=LIGER_BEST_ARGS["mode"],
        layer=LIGER_BEST_ARGS["layer"],
        seed=seed,
    )


pd.Series(
    {
        "study": OPTUNA_STUDY_NAME,
        "best_trial": OPTUNA_BEST_TRIAL,
        "best_value": OPTUNA_BEST_VALUE,
        **LIGER_BEST_ARGS,
    },
    name="best_setting",
)


study           liger_cellranger_versions_joint_expanded_metri...
best_trial                                                     99
best_value                                               0.828151
k                                                              20
value_lambda                                              2.86402
max_iters                                                      30
mode                                                         core
layer                                                      counts
Name: best_setting, dtype: object

In [5]:
def compute_joint_metrics(adata_raw, adata_int):
    return liger_opt.compute_joint_metrics(adata_raw, adata_int)


In [6]:
results = []

for seed in SEEDS:
    print(f"\n===== seed {seed} =====", flush=True)

    adata_raw = ad.read_h5ad(DATA_PATH)
    adata_int = run_liger_best(adata_raw, seed=seed)
    metric_result = compute_joint_metrics(adata_raw, adata_int)

    row = {"seed": seed, **metric_result}
    results.append(row)
    print(pd.Series(row), flush=True)

    del adata_raw, adata_int
    gc.collect()

results_df = pd.DataFrame(results)
results_df



===== seed 1 =====


/ubc/cs/research/beaver/projects/Single-cell-integration-metrics/CondaEnv/pyliger_env/lib/python3.10/site-packages/numba/np/ufunc/dufunc.py:344: NumbaWarning: Compilation requested for previously compiled argument types ((uint32,)). This has no effect and perhaps indicates a bug in the calling code (compiling a ufunc more than once for the same signature
/ubc/cs/research/beaver/projects/Single-cell-integration-metrics/CondaEnv/pyliger_env/lib/python3.10/site-packages/numba/np/ufunc/dufunc.py:344: NumbaWarning: Compilation requested for previously compiled argument types ((uint32,)). This has no effect and perhaps indicates a bug in the calling code (compiling a ufunc more than once for the same signature
/ubc/cs/research/beaver/projects/Single-cell-integration-metrics/CondaEnv/pyliger_env/lib/python3.10/site-packages/numba/np/ufunc/dufunc.py:344: NumbaWarning: Compilation requested for previously compiled argument types ((uint32,)). This has no effect and perhaps indicates a bug in the

Removing 22 genes not expressing in v6.
Index(['AMPD1', 'CRYGD', 'CPA3', 'C3orf80', 'PROM1', 'UBE2QL1', 'NR2F1',
       'STC2', 'FREM1', 'AP003096.1', 'GAL', 'GPRC5D', 'AC079949.6', 'SIX1',
       'IGHV3-64', 'ARHGDIG', 'SCNN1B', 'SPACA3', 'HOXB9', 'ONECUT2', 'SLC1A6',
       'KRTAP19-1'],
      dtype='object', name='gene')


Removing 9 genes not expressing in v8.
Index(['CRYGD', 'C3orf80', 'STC2', 'IGHV3-64', 'ARHGDIG', 'SPACA3', 'HOXB9',
       'ONECUT2', 'KRTAP19-1'],
      dtype='object', name='gene')


[LIGER PARAMS] function=optimize_ALS k=20 value_lambda=2.8640197425940483 max_iters=30 rand_seed=1


...

Cluster for leiden_opt_2.0 with leiden


resolution: 2.0, bAMI: 0.7416374467219452


optimised clustering against liger_metric_label
optimal cluster resolution: 0.2
optimal score: 0.8263234362360276
BRAS...


seed           10.000000
BAMI            0.826323
CASW            0.630669
SCGRAPHM        0.645647
BRAS            0.878266
KBET            0.898147
CPCR            0.974084
bio_score       0.700880
batch_score     0.916832
joint_score     0.787261
dtype: float64


,seed,BAMI,CASW,SCGRAPHM,BRAS,KBET,CPCR,bio_score,batch_score,joint_score
0,1,0.847043,0.691690,0.671396,0.946068,0.951120,0.998748,0.736710,0.965312,0.828151
1,2,0.817465,0.639408,0.703120,0.934449,0.958483,0.994235,0.719998,0.962389,0.816954
2,3,0.787759,0.615931,0.741547,0.934736,0.939678,0.996471,0.715079,0.956962,0.811832
3,4,0.821081,0.643358,0.722044,0.915232,0.953769,0.994072,0.728828,0.954358,0.819040
4,5,0.797611,0.634127,0.675289,0.872238,0.888118,0.935702,0.702342,0.898686,0.780880
5,6,0.816355,0.630815,0.683943,0.945073,0.928582,0.996576,0.710371,0.956744,0.808920
6,7,0.825768,0.670626,0.669183,0.928102,0.906812,0.996318,0.721859,0.943744,0.810613
7,8,0.817654,0.625171,0.678817,0.916397,0.923007,0.988967,0.707214,0.942791,0.801445
8,9,0.840098,0.662662,0.670472,0.920136,0.845064,0.969496,0.724411,0.911565,0.799273
9,10,0.826323,0.630669,0.645647,0.878266,0.898147,0.974084,0.700880,0.916832,0.787261


In [7]:
score_columns = [
    *SELECTED_METRICS,
    "bio_score",
    "batch_score",
    "joint_score",
]

summary_df = pd.DataFrame(
    {
        "mean": results_df[score_columns].mean(),
        "std": results_df[score_columns].std(ddof=0),
        "min": results_df[score_columns].min(),
        "max": results_df[score_columns].max(),
    }
)
summary_df


,mean,std,min,max
BAMI,0.819716,0.016658,0.787759,0.847043
CASW,0.644446,0.022232,0.615931,0.691690
SCGRAPHM,0.686146,0.026858,0.645647,0.741547
BRAS,0.919070,0.024186,0.872238,0.946068
KBET,0.919278,0.033672,0.845064,0.958483
CPCR,0.984467,0.018850,0.935702,0.998748
bio_score,0.716769,0.011106,0.700880,0.736710
batch_score,0.940938,0.022319,0.898686,0.965312
joint_score,0.806437,0.013751,0.780880,0.828151
